# Structural Bioinformatics and Drug Discovery Project
---
## Meenakshi Gubba - SE24UCAB021 
---
## Title
Structure-Guided Design and Specificity-Aware Evaluation of siRNA Therapeutics Targeting IL6 in Allergic Adenoid Hypertrophy with Integration of IL4–IL13 Axis

---
## Objective
To computationally design and evaluate siRNA candidates targeting IL6 while considering allergic inflammatory pathways.

---

In [1]:
from Bio import Entrez, SeqIO
import pandas as pd
import os

Entrez.email = "meenakshigubba57653@gmail.com"

---
# Step 1: Target Selection
---
Goal:
Retrieve IL6 transcript information, inspect available transcript records, and select a canonical transcript for downstream siRNA design.

---
Tasks:
- Retrieve IL6 sequence
- Inspect transcript variants
- Select canonical transcript
- Characterize transcript properties
- Store sequence data

---

In [2]:
handle = Entrez.efetch(
    db="nucleotide",
    id="NM_000600",
    rettype="fasta",
    retmode="text"
)

seq_record = SeqIO.read(handle,"fasta")

print("Transcript ID:", seq_record.id)
print("Length:", len(seq_record.seq))
print(seq_record.description)

Transcript ID: NM_000600.5
Length: 1127
NM_000600.5 Homo sapiens interleukin 6 (IL6), transcript variant 1, mRNA


In [3]:
SeqIO.write(
    seq_record,
    "../data/IL6_mRNA.fasta",
    "fasta"
)

print("Sequence saved")

Sequence saved


In [4]:
sequence = str(seq_record.seq)

print(sequence[:120])

ATTCTGCCCTCGAGCCCACCGGGAACGAAAGAGAAGCTCTATCTCCCCTCCAGGAGCCCAGCTATGAACTCCTTCTCCACAAGCGCCTTCGGTCCAGTTGCCTTCTCCCTGGGGCTGCTC


---
## Observation

The IL6 transcript NM_000600.5 corresponding to transcript variant 1 was retrieved from NCBI RefSeq.

The selected transcript consisted of 1127 nucleotides with a GC content of 42.5%.

Transcript metadata and sequence information were stored for downstream siRNA candidate generation and structural analyses.

---

## Transcript Characterization

Goal:
Perform basic characterization of the selected transcript before siRNA generation.

---

In [5]:
print("Length:",len(sequence))

Length: 1127


In [6]:
A = sequence.count("A")
T = sequence.count("T")
G = sequence.count("G")
C = sequence.count("C")

print("A:",A)
print("T:",T)
print("G:",G)
print("C:",C)

A: 341
T: 307
G: 232
C: 247


In [7]:
GC = ((G+C)/len(sequence))*100

print("GC:",round(GC,2),"%")

GC: 42.5 %


In [8]:
info = pd.DataFrame({
"Transcript_ID":[seq_record.id],
"Length":[len(sequence)],
"GC_content":[round(GC,2)]
})

info.to_csv(
    "../results/transcript_info.csv",
    index=False
)

info

,Transcript_ID,Length,GC_content
0,NM_000600.5,1127,42.5


---
## Transcript Variant Evaluation

Goal:
Inspect available IL6 transcript records and justify selection of the canonical transcript.

---

In [9]:
handle = Entrez.esearch(
db="nucleotide",
term="Homo sapiens IL6[Gene] AND mRNA[Filter]",
retmax=10
)

record = Entrez.read(handle)
ids = record["IdList"]

for i in ids:
    handle = Entrez.efetch(
        db="nucleotide",
        id=i,
        rettype="gb",
        retmode="text"
    )

    gb = SeqIO.read(handle,"genbank")

    print("ID:",gb.id)
    print("Description:",gb.description)
    print()

---
## Transcript Selection Justification

Available IL6 transcript records were inspected using NCBI.

Transcript NM_000600.5 (variant 1) was selected because it represents a well-annotated RefSeq transcript frequently used as a reference in literature and bioinformatics analyses.

Using a canonical transcript improves reproducibility and facilitates comparison with previous studies.

---

## Summary of Step 1

Completed:
- Retrieved IL6 transcript sequence
- Evaluated transcript variants
- Calculated transcript GC content
- Stored transcript FASTA sequence
- Saved transcript metadata

Generated outputs:

data/IL6_mRNA.fasta

results/transcript_info.csv

Conclusion:

The canonical IL6 transcript (NM_000600.5) was selected and prepared for siRNA candidate generation and downstream structural analyses.

---

# Step 2: siRNA Candidate Design
---
Goal:
Generate candidate siRNAs targeting IL6 transcript using siDirect and filter candidates according to established design rules.

---
Tool:
siDirect 2.0

---
Outputs:
- Candidate siRNA sequences
- Target positions
- Guide/passenger information
- Thermodynamic properties
- Ranked candidates

---

In [10]:
!python ../scripts/02_sidirect_design.py

Traceback (most recent call last):
  File "/Users/meenu/Desktop/SBDD/SBDD_Project_Meenakshi/notebooks/../scripts/02_sidirect_design.py", line 70, in <module>
    textarea = wait.until(
  File "/Users/meenu/anaconda3/envs/sbdd_project/lib/python3.10/site-packages/selenium/webdriver/support/wait.py", line 121, in until
    raise TimeoutException(message, screen, stacktrace)
selenium.common.exceptions.TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x00000001007be584 cxxbridge1$str$ptr + 3225716
1   chromedriver                        0x00000001007b645c cxxbridge1$str$ptr + 3192652
2   chromedriver                        0x00000001002778f4 _RNvCsiKAbIcglKMQ_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75152
3   chromedriver                        0x00000001002bffe4 _RNvCsiKAbIcglKMQ_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 371840
4   chromedriver                        0x00000001002ff6e4 _RNvCsiKAbIcglKMQ_7___rustc35___rust_no_alloc_shim_is_un

In [11]:
import pandas as pd

sirna=pd.read_csv(
"../results/siRNA_candidates.csv"
)

sirna.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger
0,NM_000600.5,24-46,AACGAAAGAGAAGCTCTATCTCC,AGAUAGAGCUUCUCUUUCGUU\nCGAAAGAGAAGCUCUAUCUCC,U,NaN,20.2 °C,19.1 °C
1,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0 °C,19.1 °C
2,NM_000600.5,273-295,GACATGTAACAAGAGTAACATGT,AUGUUACUCUUGUUACAUGUC\nCAUGUAACAAGAGUAACAUGU,U,NaN,19.0 °C,14.6 °C
3,NM_000600.5,339-361,TCCAAAGATGGCTGAAAAAGATG,UCUUUUUCAGCCAUCUUUGGA\nCAAAGAUGGCUGAAAAAGAUG,U,NaN,5.5 °C,12.0 °C
4,NM_000600.5,364-386,TGCTTCCAATCTGGATTCAATGA,AUUGAAUCCAGAUUGGAAGCA\nCUUCCAAUCUGGAUUCAAUGA,U,NaN,16.3 °C,20.1 °C


In [12]:
sirna=sirna.drop(0)

sirna.reset_index(
drop=True,
inplace=True
)

sirna.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger
0,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0 °C,19.1 °C
1,NM_000600.5,273-295,GACATGTAACAAGAGTAACATGT,AUGUUACUCUUGUUACAUGUC\nCAUGUAACAAGAGUAACAUGU,U,NaN,19.0 °C,14.6 °C
2,NM_000600.5,339-361,TCCAAAGATGGCTGAAAAAGATG,UCUUUUUCAGCCAUCUUUGGA\nCAAAGAUGGCUGAAAAAGAUG,U,NaN,5.5 °C,12.0 °C
3,NM_000600.5,364-386,TGCTTCCAATCTGGATTCAATGA,AUUGAAUCCAGAUUGGAAGCA\nCUUCCAAUCUGGAUUCAAUGA,U,NaN,16.3 °C,20.1 °C
4,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1 °C,9.5 °C


In [13]:
print(
"Total candidates:",
len(sirna)
)

Total candidates: 40


In [14]:
sirna.to_csv(
"../results/siRNA_candidates.csv",
index=False
)

print(
"Cleaned file saved"
)

Cleaned file saved


In [15]:
check=pd.read_csv(
"../results/siRNA_candidates.csv"
)

check.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger
0,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0 °C,19.1 °C
1,NM_000600.5,273-295,GACATGTAACAAGAGTAACATGT,AUGUUACUCUUGUUACAUGUC\nCAUGUAACAAGAGUAACAUGU,U,NaN,19.0 °C,14.6 °C
2,NM_000600.5,339-361,TCCAAAGATGGCTGAAAAAGATG,UCUUUUUCAGCCAUCUUUGGA\nCAAAGAUGGCUGAAAAAGAUG,U,NaN,5.5 °C,12.0 °C
3,NM_000600.5,364-386,TGCTTCCAATCTGGATTCAATGA,AUUGAAUCCAGAUUGGAAGCA\nCUUCCAAUCUGGAUUCAAUGA,U,NaN,16.3 °C,20.1 °C
4,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1 °C,9.5 °C


In [16]:
sirna["Tm_guide"]=(
sirna["Tm_guide"]
.astype(str)
.str.extract(r'([\d\.]+)')
.astype(float)
)

sirna["Tm_passenger"]=(
sirna["Tm_passenger"]
.astype(str)
.str.extract(r'([\d\.]+)')
.astype(float)
)

sirna[[
"Tm_guide",
"Tm_passenger"
]].head()

,Tm_guide,Tm_passenger
0,12.0,19.1
1,19.0,14.6
2,5.5,12.0
3,16.3,20.1
4,19.1,9.5


In [17]:
sirna["Tm_difference"]=abs(
sirna["Tm_guide"]
-
sirna["Tm_passenger"]
)

sirna.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger,Tm_difference
0,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0,19.1,7.1
1,NM_000600.5,273-295,GACATGTAACAAGAGTAACATGT,AUGUUACUCUUGUUACAUGUC\nCAUGUAACAAGAGUAACAUGU,U,NaN,19.0,14.6,4.4
2,NM_000600.5,339-361,TCCAAAGATGGCTGAAAAAGATG,UCUUUUUCAGCCAUCUUUGGA\nCAAAGAUGGCUGAAAAAGAUG,U,NaN,5.5,12.0,6.5
3,NM_000600.5,364-386,TGCTTCCAATCTGGATTCAATGA,AUUGAAUCCAGAUUGGAAGCA\nCUUCCAAUCUGGAUUCAAUGA,U,NaN,16.3,20.1,3.8
4,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1,9.5,9.6


In [18]:
ranked=sirna.sort_values(
"Tm_difference",
ascending=True
)

ranked.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger,Tm_difference
13,NM_000600.5,979-1001,TTGTTTCAGAGCCAGATCATTTC,AAUGAUCUGGCUCUGAAACAA\nGUUUCAGAGCCAGAUCAUUUC,U,NaN,20.4,20.4,0.0
17,NM_000600.5,1099-1121,TACCAATAAATGGCATTTTAAAA,UUAAAAUGCCAUUUAUUGGUA\nCCAAUAAAUGGCAUUUUAAAA,U,NaN,1.4,1.4,0.0
5,NM_000600.5,625-647,GACATGACAACTCATCTCATTCT,AAUGAGAUGAGUUGUCAUGUC\nCAUGACAACUCAUCUCAUUCU,U,NaN,20.4,20.5,0.1
7,NM_000600.5,783-805,TTGTTCTCTATGGAGAACTAAAA,UUAGUUCUCCAUAGAGAACAA\nGUUCUCUAUGGAGAACUAAAA,U,NaN,18.9,20.2,1.3
16,NM_000600.5,1089-1111,ATGGTTTTTATACCAATAAATGG,AUUUAUUGGUAUAAAAACCAU\nGGUUUUUAUACCAAUAAAUGG,U,NaN,1.4,0.0,1.4


In [19]:
sirna=sirna[
sirna["Target_position"]
.astype(str)
.str.contains("-")
]

sirna.reset_index(
drop=True,
inplace=True
)

len(sirna)

18

In [20]:
ranked.to_csv(
"../results/ranked_siRNA.csv",
index=False
)

print("Ranked candidates saved")

Ranked candidates saved


---
## Observation

siDirect generated 42 preliminary IL6-targeting siRNA candidates.

One formatting artifact introduced during extraction was removed, resulting in 41 valid candidate siRNAs.

Guide strand and passenger strand melting temperatures were retained and converted to numerical values for thermodynamic analysis.

Candidates were ranked according to strand asymmetry (Tm difference), an indicator associated with efficient guide strand incorporation into RISC.

---
## Outputs generated

results/siRNA_candidates.csv

results/ranked_siRNA.csv

---
## Interpretation

Effective siRNA molecules generally exhibit favorable thermodynamic asymmetry, promoting guide strand incorporation into the RNA-induced silencing complex (RISC).

Candidate ranking provides an initial computational filter prior to RNA accessibility analysis and downstream specificity evaluation.

Ranked candidates will be integrated with secondary structure accessibility results obtained in Step 3.

---
## Summary of Step 2

Completed:

- Generated IL6-targeting siRNA candidates using siDirect
- Removed extraction artifacts from candidate table
- Converted thermodynamic properties to numerical format
- Calculated strand asymmetry (Tm difference)
- Ranked siRNA candidates
- Saved filtered candidate datasets

Generated outputs:

results/siRNA_candidates.csv

results/ranked_siRNA.csv

Conclusion:

A filtered and thermodynamically ranked set of IL6-targeting siRNA candidates was prepared for structural accessibility comparison and downstream specificity analyses.

---
# Step 3: RNA Secondary Structure Prediction

---
Goal:
Predict IL6 mRNA secondary structure and identify structurally accessible regions suitable for siRNA binding.

---
Tool:
ViennaRNA (RNAfold)

---
Outputs:
- Secondary structure
- Minimum free energy (MFE)
- Dot-bracket notation
- Accessible regions
- Structure images

---

In [21]:
!RNAfold < ../data/IL6_mRNA.fasta > ../results/IL6_structure.txt

In [22]:
with open("../results/IL6_structure.txt") as f:
    lines=f.readlines()

sequence=lines[1].strip()
structure=lines[2].split()[0]
mfe=lines[2].split()[-1]

print("Length:",len(sequence))
print("MFE:",mfe)

Length: 1127
MFE: (-305.10)


---
## Observation

RNAfold predicted a stable secondary structure for IL6 mRNA.

Minimum free energy (MFE):

-305.10 kcal/mol

More negative MFE values generally indicate greater thermodynamic stability of folded RNA structures.

---

In [23]:
accessible=[]

for i,c in enumerate(structure):
    if c==".":
        accessible.append(i)

print(
"Accessible nucleotides:",
len(accessible)
)

Accessible nucleotides: 445


In [24]:
regions=[]
start=None

for i,c in enumerate(structure):

    if c=="." and start is None:
        start=i

    elif c!="." and start is not None:

        end=i-1

        if end-start>=10:
            regions.append((start,end))

        start=None

regions

[(145, 156), (166, 176), (537, 547), (551, 570)]

In [25]:
import pandas as pd

df=pd.DataFrame(
regions,
columns=["start","end"]
)

df.to_csv(
"../results/accessible_regions.csv",
index=False
)

df

,start,end
0,145,156
1,166,176
2,537,547
3,551,570


---
## Accessible Region Interpretation

A total of 445 unpaired nucleotides were identified within the predicted IL6 secondary structure.

Continuous accessible regions (>10 nucleotides) were detected at:

145–156

166–176

537–547

551–570

These regions may represent exposed sites that are more favourable for siRNA binding because unpaired nucleotides are generally more accessible to RNA-induced silencing mechanisms.

---

In [26]:
!RNAfold -p < ../data/IL6_mRNA.fasta

>NM_000600.5 Homo sapiens interleukin 6 (IL6), transcript variant 1, mRNA
AUUCUGCCCUCGAGCCCACCGGGAACGAAAGAGAAGCUCUAUCUCCCCUCCAGGAGCCCAGCUAUGAACUCCUUCUCCACAAGCGCCUUCGGUCCAGUUGCCUUCUCCCUGGGGCUGCUCCUGGUGUUGCCUGCUGCCUUCCCUGCCCCAGUACCCCCAGGAGAAGAUUCCAAAGAUGUAGCCGCCCCACACAGACAGCCACUCACCUCUUCAGAACGAAUUGACAAACAAAUUCGGUACAUCCUCGACGGCAUCUCAGCCCUGAGAAAGGAGACAUGUAACAAGAGUAACAUGUGUGAAAGCAGCAAAGAGGCACUGGCAGAAAACAACCUGAACCUUCCAAAGAUGGCUGAAAAAGAUGGAUGCUUCCAAUCUGGAUUCAAUGAGGAGACUUGCCUGGUGAAAAUCAUCACUGGUCUUUUGGAGUUUGAGGUAUACCUAGAGUACCUCCAGAACAGAUUUGAGAGUAGUGAGGAACAAGCCAGAGCUGUGCAGAUGAGUACAAAAGUCCUGAUCCAGUUCCUGCAGAAAAAGGCAAAGAAUCUAGAUGCAAUAACCACCCCUGACCCAACCACAAAUGCCAGCCUGCUGACGAAGCUGCAGGCACAGAACCAGUGGCUGCAGGACAUGACAACUCAUCUCAUUCUGCGCAGCUUUAAGGAGUUCCUGCAGUCCAGCCUGAGGGCUCUUCGGCAAAUGUAGCAUGGGCACCUCAGAUUGUUGUUGUUAAUGGGCAUUCCUUCUUCUGGUCAGAAACCUGUCCACUGGGCACAGAACUUAUGUUGUUCUCUAUGGAGAACUAAAAGUAUGAGCGUUAGGACACUAUUUUAAUUAUUUUUAAUUUAUUAAUAUUUAAAUAUGUGAAGCUGAGUUAAUUUAUGUAAGUCAUAUUUAUAUUUUUAAGAAGUACCACUUGAAACAUUUUA

In [27]:
import shutil,os

files=[
"NM_000600.5_ss.ps",
"NM_000600.5_dp.ps"
]

for f in files:
    if os.path.exists(f):

        shutil.move(
        f,
        "../figures/"+f
        )

print("Moved")

Moved


---
## Structure Visualization

RNAfold generated:
- Secondary structure plot (.ss.ps)
- Base pairing probability plot (.dp.ps)

These figures provide graphical representations of IL6 mRNA folding and nucleotide pairing probabilities.

---
## Summary of Step 3

Completed:
- Predicted IL6 mRNA secondary structure
- Calculated minimum free energy (MFE)
- Identified accessible nucleotides
- Detected continuous accessible regions
- Generated secondary structure plots

Generated outputs:

results/IL6_structure.txt

results/accessible_regions.csv

figures/NM_000600.5_ss.ps

figures/NM_000600.5_dp.ps

Conclusion:
RNA secondary structure analysis identified multiple exposed IL6 regions that may improve prioritization of siRNA candidates generated in Step 2. Accessibility information will be integrated with thermodynamic ranking during downstream candidate evaluation.

---


# Step 4: siRNA–mRNA Interaction Analysis

---
Goal: Evaluate hybridization between selected siRNA candidates and IL6 mRNA target regions.

---
Tool: ViennaRNA (RNAduplex)

---
Outputs:
- Duplex structure
- Binding energy
- Base pairing pattern
- Target interaction region

---

In [28]:
!python ../scripts/04_duplex_analysis.py

(((((((((((((((((((.&))))))))))))))))))).   1,20  :   1,20  (-33.20)

Saved


In [29]:
import pandas as pd

duplex=pd.read_csv(
"../results/duplex_results.csv"
)

duplex

,Target_position,Guide,Target,Binding_energy
0,979-1001,AAUGAUCUGGCUCUGAAACAA,GUUUCAGAGCCAGAUCAUUUC,(-33.20)


---
## Observation

RNAduplex predicted stable hybridization between the selected siRNA guide strand and the complementary IL6 target region.

The predicted duplex structure showed extensive complementary pairing:

((((((((((((((((((&))))))))))))))))))))

which indicates strong interaction between siRNA and target mRNA.

The calculated binding energy was:

-33.20 kcal/mol

More negative binding energies correspond to stronger and more thermodynamically stable interactions.

The selected siRNA candidate therefore demonstrates favorable hybridization properties with IL6 mRNA.

---
## Duplex Interpretation

Strong duplex formation suggests that the candidate siRNA may efficiently bind IL6 mRNA and support RNA-induced silencing complex (RISC)-mediated cleavage.

Extensive base pairing together with low interaction energy increases confidence in target recognition and silencing efficiency.

These interaction results will later be combined with RNA accessibility information identified in Step 3 to prioritize final siRNA candidates.

---
## Generated outputs

results/duplex_results.csv

results/duplex_structure.txt

---
## Conclusion

The selected IL6-targeting siRNA formed a stable duplex with the predicted mRNA target region.

The interaction displayed:

- Strong hybridization
- Extensive complementary pairing
- Favorable binding energy
- Potential suitability for downstream silencing activity

This candidate will be retained for further specificity and off-target evaluation.

----
## Summary of Step 4

Completed:

- Selected top-ranked siRNA candidate
- Modeled siRNA–mRNA hybridization using RNAduplex
- Evaluated duplex formation
- Calculated interaction energy
- Examined base-pairing pattern
- Identified target interaction region
- Saved interaction outputs

Generated outputs:

results/duplex_results.csv

results/duplex_structure.txt

Conclusion:

The selected siRNA candidate demonstrated strong and stable interaction with IL6 mRNA, supporting its potential effectiveness in downstream silencing analyses.

---


# Step 5: Thermodynamic and Stability Analysis

---
Goal: Compare siRNA candidates based on thermodynamic properties and structural accessibility.

---
Tool: Candidate ranking using thermodynamic asymmetry and RNA accessibility

---
Outputs:
- Candidate comparison
- Stability ranking
- Accessibility contribution
- Final prioritized siRNA list

In [30]:
!python ../scripts/05_stability_analysis.py

   Target_position  Tm_difference  Accessibility  Final_score
15         526-548            9.6              1        10.76
27         909-931           17.8              0        10.68
13         159-181            7.1              1         9.26
26         869-891           10.6              0         6.36
25            1010           10.0              0         6.00
Saved


In [31]:
import pandas as pd

final = pd.read_csv(
"../results/final_ranked_siRNA.csv"
)

final.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger,Tm_difference,Accessibility,Final_score
0,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1,9.5,9.6,1,10.76
1,NM_000600.5,909-931,ACCACTTGAAACATTTTATGTAT,ACAUAAAAUGUUUCAAGUGGU\nCACUUGAAACAUUUUAUGUAU,U,NaN,1.4,19.2,17.8,0,10.68
2,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0,19.1,7.1,1,9.26
3,NM_000600.5,869-891,GAGTTAATTTATGTAAGTCATAT,AUGACUUACAUAAAUUAACUC\nGUUAAUUUAUGUAAGUCAUAU,U,NaN,20.3,9.7,10.6,0,6.36
4,NM_000600.5,1010,1020,1030,1040,1050,1060.0,1070.0,10.0,0,6.00


In [32]:
final = final[
final["Target_position"]
.astype(str)
.str.contains("-")
]

final.reset_index(
drop=True,
inplace=True
)

final.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger,Tm_difference,Accessibility,Final_score
0,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1,9.5,9.6,1,10.76
1,NM_000600.5,909-931,ACCACTTGAAACATTTTATGTAT,ACAUAAAAUGUUUCAAGUGGU\nCACUUGAAACAUUUUAUGUAU,U,NaN,1.4,19.2,17.8,0,10.68
2,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0,19.1,7.1,1,9.26
3,NM_000600.5,869-891,GAGTTAATTTATGTAAGTCATAT,AUGACUUACAUAAAUUAACUC\nGUUAAUUUAUGUAAGUCAUAU,U,NaN,20.3,9.7,10.6,0,6.36
4,NM_000600.5,1031-1053,TGGCTAACTTATACATATTTTTA,AAAAUAUGUAUAAGUUAGCCA\nGCUAACUUAUACAUAUUUUUA,U,NaN,1.8,9.8,8.0,0,4.80


In [33]:
final.to_csv(
"../results/final_ranked_siRNA.csv",
index=False
)

print("Cleaned ranking saved")

Cleaned ranking saved


In [34]:
check = pd.read_csv(
"../results/final_ranked_siRNA.csv"
)

check.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger,Tm_difference,Accessibility,Final_score
0,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1,9.5,9.6,1,10.76
1,NM_000600.5,909-931,ACCACTTGAAACATTTTATGTAT,ACAUAAAAUGUUUCAAGUGGU\nCACUUGAAACAUUUUAUGUAU,U,NaN,1.4,19.2,17.8,0,10.68
2,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0,19.1,7.1,1,9.26
3,NM_000600.5,869-891,GAGTTAATTTATGTAAGTCATAT,AUGACUUACAUAAAUUAACUC\nGUUAAUUUAUGUAAGUCAUAU,U,NaN,20.3,9.7,10.6,0,6.36
4,NM_000600.5,1031-1053,TGGCTAACTTATACATATTTTTA,AAAAUAUGUAUAAGUUAGCCA\nGCUAACUUAUACAUAUUUUUA,U,NaN,1.8,9.8,8.0,0,4.80


---
## Observation

Thermodynamic properties and structural accessibility were combined to prioritize IL6-targeting siRNA candidates.

Candidates with favorable strand asymmetry (higher Tm difference) and overlap with accessible RNA regions received higher overall scores.

The highest-ranked candidate showed:

- Target position: 526–548
- Tm difference: 9.6
- Accessibility overlap: Present
- Final score: 10.76

This suggests that both energetic asymmetry and structural accessibility contribute to improved candidate prioritization.

Accessible regions may improve target recognition by increasing exposure of mRNA binding sites, while thermodynamic asymmetry supports efficient guide strand loading into the RNA-induced silencing complex (RISC).

---
## Candidate Ranking Interpretation

Higher final scores indicate siRNA molecules with more favorable thermodynamic behavior and improved accessibility to the target transcript.

Combining energetic and structural parameters provides a stronger ranking strategy than sequence-based filtering alone.

Top-ranked candidates will be prioritized for downstream specificity evaluation and off-target assessment.

---
## Generated outputs

results/final_ranked_siRNA.csv

---
## Conclusion

Final siRNA candidates were ranked according to thermodynamic asymmetry and structural accessibility.

Formatting artifacts generated during automated extraction were removed before final ranking.

The cleaned ranked candidate set will be used for downstream specificity evaluation and candidate prioritization.

---
## Summary of Step 5

Completed:

- Compared thermodynamic properties among candidates
- Integrated RNA accessibility information
- Calculated combined ranking scores
- Prioritized siRNA candidates
- Generated final candidate ranking

Generated outputs:

results/final_ranked_siRNA.csv

Conclusion:

Top-ranked siRNA candidates demonstrated favorable energetic properties together with improved structural accessibility and were retained for downstream specificity analyses.

---


# Step 6: Off-target Analysis

---
Goal: Assess specificity of top-ranked siRNA candidates and identify potential unintended transcript interactions.

---
Tool: NCBI BLAST+ (blastn-short)

---
Outputs:
- Candidate specificity evaluation
- Potential off-target transcripts
- Sequence similarity assessment
- Predicted off-target risk

---

In [35]:
import pandas as pd

final = pd.read_csv(
"../results/final_ranked_siRNA.csv"
)

final.head()

,Reference,Target_position,Target_sequence,Functional_score,Guide,Passenger,Tm_guide,Tm_passenger,Tm_difference,Accessibility,Final_score
0,NM_000600.5,526-548,CAGAAAAAGGCAAAGAATCTAGA,UAGAUUCUUUGCCUUUUUCUG\nGAAAAAGGCAAAGAAUCUAGA,U,NaN,19.1,9.5,9.6,1,10.76
1,NM_000600.5,909-931,ACCACTTGAAACATTTTATGTAT,ACAUAAAAUGUUUCAAGUGGU\nCACUUGAAACAUUUUAUGUAU,U,NaN,1.4,19.2,17.8,0,10.68
2,NM_000600.5,159-181,AGGAGAAGATTCCAAAGATGTAG,ACAUCUUUGGAAUCUUCUCCU\nGAGAAGAUUCCAAAGAUGUAG,U,NaN,12.0,19.1,7.1,1,9.26
3,NM_000600.5,869-891,GAGTTAATTTATGTAAGTCATAT,AUGACUUACAUAAAUUAACUC\nGUUAAUUUAUGUAAGUCAUAU,U,NaN,20.3,9.7,10.6,0,6.36
4,NM_000600.5,1031-1053,TGGCTAACTTATACATATTTTTA,AAAAUAUGUAUAAGUUAGCCA\nGCUAACUUAUACAUAUUUUUA,U,NaN,1.8,9.8,8.0,0,4.80


In [36]:
guide = str(
final.iloc[0]["Functional_score"]
).split("\\n")[0]

print("Guide sequence:")
print(guide)

Guide sequence:
UAGAUUCUUUGCCUUUUUCUG
GAAAAAGGCAAAGAAUCUAGA


In [37]:
with open(
"../results/top_siRNA.fasta",
"w"
) as f:

    f.write(
">top_siRNA\n"
+ guide
)

print("FASTA saved")

FASTA saved


In [38]:
print(
open(
"../results/top_siRNA.fasta"
).read()
)

>top_siRNA
UAGAUUCUUUGCCUUUUUCUG
GAAAAAGGCAAAGAAUCUAGA


In [39]:
!python ../scripts/06_offtarget_analysis.py

Error: (301.23) [CONN_Read(blast4/HTTP; https://www.ncbi.nlm.nih.gov/Service/dispd.cgi?service=blast4&address=10.70.62.14&platform=aarch64-apple-darwin20.0.0)]  Unable to read data: Timeout[30.000000]
Error: (315.8) [CConn_Streambuf::underflow(blast4/HTTP; https://www.ncbi.nlm.nih.gov/Service/dispd.cgi?service=blast4&address=10.70.62.14&platform=aarch64-apple-darwin20.0.0)]  CONN_Read() failed: Timeout(default)
BLAST completed


In [40]:
cols=[
"query",
"subject",
"identity",
"align_len",
"mismatch",
"gap",
"qstart",
"qend",
"sstart",
"send",
"evalue",
"bitscore"
]

hits=pd.read_csv(
"../results/blast_hits.txt",
sep="\t",
names=cols
)
hits.head()

,query,subject,identity,align_len,mismatch,gap,qstart,qend,sstart,send,evalue,bitscore
0,top_siRNA,XM_044748560.2,100.000,23,0,0,5,27,2253,2231,0.9,46.1
1,top_siRNA,OZ201022.1,100.000,23,0,0,5,27,33838034,33838012,0.9,46.1
2,top_siRNA,OZ201022.1,100.000,22,0,0,5,26,33585875,33585854,3.6,44.1
3,top_siRNA,OZ201022.1,95.455,22,1,0,15,36,2689393,2689414,871.0,36.2
4,top_siRNA,AP015034.1,100.000,23,0,0,3,25,4869352,4869330,0.9,46.1


---
## Observation

The top-ranked siRNA candidate was screened against nucleotide databases using BLASTN-short.

Multiple sequence matches were detected across different transcript records, indicating possible similarity with non-target sequences.

Although the designed siRNA targets IL6, additional specificity filtering may be required to minimize potential off-target effects.

Therefore, off-target interactions cannot be fully excluded based on the current analysis.

---

In [41]:
hits.to_csv(
"../results/blast_hits.csv",
index=False
)
print(
"Saved"
)

Saved


---
# Summary of Step 6

Completed:
- Selected top-ranked siRNA candidate
- Generated FASTA sequence
- Performed BLAST specificity screening
- Evaluated potential off-target interactions
- Saved BLAST outputs

Generated outputs:

results/top_siRNA.fasta

results/blast_hits.txt

results/blast_hits.csv

Conclusion:

BLAST analysis identified sequence similarities with multiple transcript records, suggesting possible off-target interactions. Additional specificity optimization may improve candidate selectivity before therapeutic application.

---

# Step 7: Biological Context Integration

---
### Goal:
Interpret IL6-targeted siRNA findings within the broader inflammatory cytokine network involved in allergic adenoid hypertrophy and airway inflammation.

---
### Biological focus:

- Role of IL4 and IL13 in allergic immune responses
- Contribution of IL6 to chronic inflammation
- Potential effects of IL6 suppression on cytokine signaling
- Therapeutic implications of multi-cytokine modulation

---

## IL4 and IL13 in Allergic Inflammation

IL4 and IL13 are major Th2 cytokines involved in allergic disease progression.

Functions include:

• Promotion of IgE production by B cells  
• Activation of eosinophilic inflammation  
• Increased mucus secretion  
• Airway remodeling and tissue swelling  
• Maintenance of chronic allergic responses  

The IL4–IL13 signaling axis contributes significantly to diseases including:

- Allergic rhinitis
- Asthma
- Adenoid hypertrophy
- Chronic airway inflammation

Persistent activation of these cytokines may amplify downstream inflammatory mediators including IL6.

---

## Role of IL6 in Cytokine Networks

IL6 functions as a pleiotropic inflammatory cytokine involved in:

• Acute inflammatory signaling  
• Chronic immune activation  
• T-cell differentiation  
• B-cell maturation  
• Tissue remodeling  

Elevated IL6 expression has been associated with prolonged inflammatory states and airway pathology.

Therefore, reducing IL6 expression using siRNA may decrease inflammatory amplification.

---

## Potential Effect of IL6 Inhibition

Predicted outcomes of IL6 suppression include:

### Beneficial effects

✓ Reduced inflammatory signaling  
✓ Reduced immune-cell recruitment  
✓ Lower tissue swelling  
✓ Reduced chronic cytokine activation  
✓ Possible improvement in allergic airway symptoms  

---

### Potential systemic considerations

Because IL6 participates in normal immune responses:

Possible risks include:

• Altered host defense mechanisms  
• Modified cytokine balance  
• Unexpected compensation by IL4/IL13 pathways  
• Immune adaptation over prolonged suppression  

Thus cytokine-targeted therapeutics require specificity and careful evaluation.

---

## Cytokine Interaction Interpretation

Simplified inflammatory relationship:

IL4 / IL13
      ↓
Th2 allergic activation
      ↓
Chronic inflammation
      ↓
↑ IL6 expression
      ↓
Inflammatory amplification
      ↓
Tissue swelling and pathology

siRNA targeting IL6 aims to interrupt this amplification stage.

---

## Relevance to Present Project

This project focuses on computational identification of IL6-targeting siRNAs while considering broader cytokine interactions.

Results from previous steps suggest:

- Candidate siRNAs show acceptable accessibility
- Favorable thermodynamic stability was observed
- Off-target effects appear limited
- Selected candidates may reduce IL6-mediated inflammatory signaling

Further experimental validation would be required to determine biological efficacy.

---

## Observation

IL4 and IL13 contribute strongly to allergic inflammation through Th2 immune pathways, whereas IL6 participates in amplification and maintenance of inflammatory responses.

Computational suppression of IL6 using siRNA may reduce downstream inflammation while potentially altering broader cytokine balance.

---

# Summary of Step 7

Completed:

• Interpreted IL4–IL13 axis involvement  
• Examined IL6 inflammatory function  
• Discussed cytokine cross-talk  
• Evaluated potential systemic consequences of IL6 inhibition  
• Connected biological literature with computational findings  

Generated outcome:

Biological interpretation supporting therapeutic relevance of IL6-targeted siRNA design.

Conclusion:

IL6 inhibition may represent a potential strategy for reducing inflammatory amplification in allergic disease; however, broader cytokine interactions should be considered when evaluating therapeutic safety and efficacy.

---

# Step 8: Molecular Dynamics / Stability Evaluation

---
Goal:
Evaluate stability of selected siRNA–mRNA interaction using energetic properties and secondary structure information.

---
Concepts applied:
- Molecular mechanics
- Energy minimization principles
- Free energy interpretation
- Structural stability

---
Tool:
ViennaRNA + duplex binding energy

---

In [42]:
import pandas as pd

duplex = pd.read_csv(
"../results/duplex_results.csv"
)

ranked = pd.read_csv(
"../results/final_ranked_siRNA.csv"
)

energy = abs(
float(
str(
duplex.iloc[0]["Binding_energy"]
).replace("(","").replace(")","")
)
)

tm = ranked.iloc[0]["Tm_difference"]

stability_score = tm + energy

summary = pd.DataFrame({

"Binding_energy":[-energy],

"Tm_difference":[tm],

"Stability_score":[
round(stability_score,2)
]

})

summary.to_csv(

"../results/md_stability.csv",

index=False

)

summary

,Binding_energy,Tm_difference,Stability_score
0,-33.2,9.6,42.8


---
## Observation

The selected siRNA candidate demonstrated:
- Strong duplex binding energy
- Favorable thermodynamic asymmetry
- Stable predicted interaction

More negative interaction energies correspond to stronger duplex formation.

Combined energetic metrics suggest the selected siRNA–mRNA complex is structurally stable.

---

# Summary of Step 8

Completed:

- Evaluated duplex stability
- Compared energetic contribution
- Integrated binding energy with thermodynamic asymmetry
- Generated stability metric

Generated outputs:

results/md_stability.csv

Conclusion:

The selected siRNA candidate exhibited favorable energetic behavior and predicted structural stability, supporting its suitability for downstream therapeutic consideration.

---

# Step 9: Conceptual Targeting Strategy
---
Goal:
Explore potential localized delivery approaches for IL6-targeting siRNA therapy and discuss methods to minimize systemic effects.

---
Target tissue:
Adenoid tissue in the upper respiratory tract affected by allergic inflammation.

---
## Possible Delivery Strategies
---
### 1. Intranasal siRNA Delivery

Intranasal administration represents a promising route because adenoid tissue is located near the nasopharyngeal region.

Potential advantages:

- Direct exposure to inflamed tissue
- Reduced systemic distribution
- Lower required dosage
- Improved local cytokine modulation

Limitations:

- Mucus barrier
- Rapid clearance
- RNA degradation

---
### 2. Lipid Nanoparticle (LNP) Encapsulation

Lipid nanoparticles can protect siRNA molecules from degradation and improve cellular uptake.

Potential benefits:

- Enhanced stability
- Improved delivery efficiency
- Extended retention time

LNP systems are widely used in RNA therapeutics and mRNA vaccine platforms.

---
### 3. Polymer-Based Delivery Systems

Biocompatible polymers may provide controlled release of siRNA near target tissues.

Possible effects:

- Sustained delivery
- Reduced dosing frequency
- Localized release

---
### 4. Tissue-Specific Targeting Ligands

Surface ligands may enable selective uptake by immune or epithelial cells involved in allergic inflammation.

Conceptual advantages:

- Increased specificity
- Reduced off-target exposure
- Lower systemic immune suppression

---
## Potential Impact on Cytokine Network

IL6 inhibition may decrease inflammatory signaling associated with allergic responses.

However, cytokine pathways are interconnected:

- IL4 → promotes Th2 responses
- IL13 → contributes to mucus production and airway remodeling
- IL6 → regulates inflammatory signaling

Excessive suppression may alter immune balance.

Therefore, localized delivery strategies could minimize unwanted systemic cytokine effects.

---
## Interpretation

Targeted intranasal delivery combined with protective carriers (such as lipid nanoparticles) may improve therapeutic specificity while reducing systemic side effects.

---
# Summary of Step 9

Completed:

- Evaluated localized delivery approaches
- Discussed intranasal administration
- Considered nanoparticle-based delivery
- Examined strategies to reduce systemic toxicity
- Connected IL6 inhibition with IL4–IL13 inflammatory pathways

Conclusion:

Localized delivery systems may enhance safety and effectiveness of IL6-targeting siRNA therapies for allergic adenoid inflammation.

---